# Continual learning without task instructions: "all adapters" against "zones"

**Nothing has to be edited.** Five SuperNI tasks are learned one after another with t5-base, with the task definition removed from every
input. Each task gets its own low-rank adapter (800 training examples, 10 epochs, learning rate 3e-4). The same adapters are then used in two ways:
**all** (all adapters switched on) and **zones** (each task claims a zone built from the inputs of its training examples; an input is answered
with the adapters up to the oldest task whose zone holds it). After the last task the test inputs (100 per task) are answered again and compared
with the answers right after learning. Two runs, seeds 42 and 43, one per GPU, in parallel.

**How to use it on Kaggle:** right-hand panel, *Session options*: Accelerator **GPU T4 x2**, **Internet on** (needed for the data and the
t5-base weights; if it is off the notebook stops at once and says so). Then **Save Version -> Save & Run All**.

**How long:** the earlier notebook needed about 45 minutes per job on a T4; the two jobs here run at the same time, so expect about
**45 to 60 minutes** in total (an estimate, not measured on a GPU for this script).

**The result** is the file `/kaggle/working/no_instruction_small_files.zip` in the notebook's output (two JSON result files and two logs, no model weights).
A small table is also printed at the end.

In [ ]:
import os, sys, json, time, shutil, subprocess, threading, zipfile

NB_TEST = os.environ.get("NI_NB_TEST") == "1"          # only to test this notebook on a CPU; has no effect on Kaggle
OUT = os.environ.get("NI_NB_OUT") or "/kaggle/working/no_instruction"
ZIP = "/kaggle/working/no_instruction_small_files.zip" if not NB_TEST else OUT.rstrip("/") + "_small_files.zip"
CODE = os.path.join(OUT, "code")
LOGS = os.path.join(OUT, "logs")
RESULTS = os.path.join(OUT, "results")
WORK = os.environ.get("NI_NB_WORK") or ("/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp") + "/no_instruction_work"
CFG = dict(
    model="t5-base", seeds=[42, 43], train=800, test=100, epochs=10, lr="3e-4",
    repo_url="https://github.com/liangyanshuo/gainlora",
    repo_commit="e9d0d700538c8d12892f7fd52b216291f7e42acc",     # the version of the data repository used for the earlier runs
    progress_every_s=60,
)
if NB_TEST:
    CFG.update(model="t5-small", seeds=[42], train=16, test=4, epochs=1, lr="3e-4", progress_every_s=20)
for d in (CODE, LOGS, RESULTS, WORK):
    os.makedirs(d, exist_ok=True)
print("results are written to:", OUT)

## The code
The next cell holds the project's two Python files (copies, not edited); the cell after it writes them to disk.

In [ ]:
FILES = globals().get("FILES", {})
FILES["certified_switch.py"] = r'''"""The certified switch as stand-alone code, for any model that has an embedding table and linear layers.

Nothing here depends on one architecture or on the code of another paper. Three parts:

  Zones          decides, from the frozen word-vectors of an input, which task the input is given to.
                 Each task has a protected zone that is never changed; the oldest zone that contains
                 an input owns it for ever; an input outside every zone goes to the nearest zone.
  RoutedLoRA     one low-rank adapter per task on a linear layer. Older adapters are frozen. For every
                 input only the adapters of tasks 1..route are switched on.
  ContinualModel puts both on a Hugging Face model (encoder-decoder or decoder-only), learns tasks one
                 after another and answers inputs.

Why an answer inside a zone can never change: the route of such an input is the oldest zone that
contains it. Later tasks only add newer zones, so the route stays the same, and the adapters it
switches on are frozen. This holds for any model and any later tasks.

Two rules are offered (`rule=` of ContinualModel):
  "certified"    the rule above. A new task whose instruction lies inside an older zone is refused on those
                 inputs (they stay with the older task); `learn` reports how many.
  "exact_first"  an input that carries the instruction of a task token for token goes to that task (the
                 oldest, if several tasks have the same instruction); otherwise as above. A sibling of an old
                 task can then be learned. The guarantee is weaker: an answer inside a zone stays the same
                 unless a task is learned later whose instruction that input carries exactly. An input that
                 carries a learned instruction exactly keeps its answer without conditions.
On tasks whose instructions do not collide the two rules give the same routes.

What it needs: every task's inputs start with the same text (its instruction). That opening is
what the zone is built from.
"""
import math

import torch
import torch.nn as nn


# ----------------------------------------------------------------------------------------------
# 1. Which task does an input belong to?
# ----------------------------------------------------------------------------------------------
def common_prefix(rows):
    """Number of tokens that all rows share at their start."""
    n = 0
    while all(len(r) > n for r in rows) and len({r[n] for r in rows}) == 1:
        n += 1
    return n


class Zones:
    """Protected zones built from a frozen embedding table.

    table    the model's input embedding matrix (vocabulary x dimensions); it is only read
    kappa    radius of a zone = kappa x distance from its signature to the nearest signature known
             when the task is added (unrelated tasks and earlier tasks)
    a        a token counts with weight a / (a + its share among the tokens of unrelated text)
    """

    def __init__(self, table, pad_id, kappa=0.5, a=1e-3, head=64):
        self.table = table.detach().double()
        self.pad, self.kappa, self.a, self.head = pad_id, kappa, a, head
        self.weight = None
        self.tasks = []                                  # one dict per task, oldest first; never changed

    def _mean(self, rows, n=None):
        """Weighted average word-vector of the first n tokens of every row (all tokens if n is None)."""
        out = []
        for r in rows:
            ids = torch.as_tensor(r[:n] if n else r, dtype=torch.long, device=self.table.device)
            w = (ids != self.pad).double()
            if self.weight is not None:
                w = w * self.weight[ids]
            out.append((self.table[ids] * w[:, None]).sum(0) / w.sum().clamp(min=1e-12))
        return torch.stack(out)

    @staticmethod
    def _whitener(x):
        """Centre and rescale so that unrelated inputs vary by about the same amount in every direction."""
        mu = x.mean(0)
        cov = (x - mu).T @ (x - mu) / (x.shape[0] - 1)
        cov = 0.5 * cov + 0.5 * torch.eye(cov.shape[0], dtype=cov.dtype, device=cov.device) * torch.trace(cov) / cov.shape[0]
        w, v = torch.linalg.eigh(cov)
        return mu, v @ torch.diag(w ** -0.5) @ v.T

    def fit_background(self, unrelated_tasks):
        """unrelated_tasks: for each unrelated task, a list of token-id rows (its inputs, instruction included).
        Sets the token weights, the scale of distances and the signatures of the unrelated tasks. Done once."""
        rows = [r for task in unrelated_tasks for r in task]
        if self.a > 0:
            counts = torch.zeros(self.table.shape[0], dtype=torch.double, device=self.table.device)
            for r in rows:
                ids = torch.as_tensor([i for i in r if i != self.pad], dtype=torch.long, device=self.table.device)
                counts.index_add_(0, ids, torch.ones(len(ids), dtype=torch.double, device=self.table.device))
            self.weight = self.a / (self.a + counts / counts.sum())
            self.weight[self.pad] = 0.0
        self.mu_head, self.w_head = self._whitener(self._mean(rows, self.head))
        self.mu_all, self.w_all = self._whitener(self._mean(rows))
        self.known = torch.stack([self._signature(task[0][:common_prefix(task[:20])]) for task in unrelated_tasks])

    def _signature(self, opening):
        return (self._mean([opening])[0] - self.mu_head) @ self.w_head

    def add_task(self, train_rows, opening_len=None):
        """Make the zone of a new task from its training inputs. Returns how many of them were already inside
        an older zone (those inputs stay with the older task: the price of the guarantee, visible at once)."""
        n = opening_len or common_prefix(train_rows)
        assert n >= 3, "the task's inputs do not start with a common instruction"
        blocked = int(self.route(train_rows)[1].sum()) if self.tasks else 0
        sig = self._signature(train_rows[0][:n])
        known = torch.cat([self.known] + [t["sig"][None] for t in self.tasks])
        centre = ((self._mean(train_rows) - self.mu_all) @ self.w_all).mean(0)
        self.tasks.append({"n": n, "sig": sig, "rho": self.kappa * float((known - sig).norm(dim=1).min()), "centre": centre,
                           "opening": list(train_rows[0][:n])})
        return blocked

    def route(self, rows, rule="certified"):
        """For every row: its route (1-based task number; adapters 1..route are used) and whether it is certified.
        rule: 'certified' (oldest zone, else nearest zone), 'exact_first' (the oldest task whose instruction the
        input carries token for token, else as 'certified'), 'nearest' (nearest zone), 'centre' (nearest centre of
        the whole input), 'all' (the newest task: every adapter on)."""
        T = len(self.tasks)
        route = torch.full((len(rows),), T, dtype=torch.long)
        cert = torch.zeros(len(rows), dtype=torch.bool)
        if rule == "all" or T == 0:
            return route, cert
        if rule == "centre":
            whole = (self._mean(rows) - self.mu_all) @ self.w_all
            return torch.stack([(whole - t["centre"]).norm(dim=1) for t in self.tasks], 1).argmin(1).cpu() + 1, cert
        means = {n: self._mean(rows, n) for n in {t["n"] for t in self.tasks}}
        rel = torch.stack([(((means[t["n"]] - self.mu_head) @ self.w_head) - t["sig"]).norm(dim=1) / t["rho"]
                           for t in self.tasks], 1).cpu()        # 1 or less: inside that zone
        # A zone of radius zero (its signature equals a known one) holds only inputs at distance zero: 0 / 0 counts as
        # inside (0), any other distance as outside (infinity). Ties go to the older task (argmin takes the first).
        rel = torch.where(torch.isnan(rel), torch.zeros_like(rel), rel)
        nearest = rel.argmin(1) + 1
        if rule in ("certified", "exact_first"):
            for k in range(T):                               # oldest first: an older claim is never overruled
                new = (rel[:, k] <= 1.0) & ~cert
                route[new] = k + 1
                cert |= new
        route = torch.where(cert, route, nearest)
        if rule == "exact_first":
            for i, row in enumerate(rows):                   # an exact claim comes before every zone; the oldest exact claim wins
                for k, t in enumerate(self.tasks):
                    if len(row) >= t["n"] and list(row[:t["n"]]) == t["opening"]:
                        route[i], cert[i] = k + 1, True
                        break
        return route, cert


# ----------------------------------------------------------------------------------------------
# 2. One adapter per task, switched on and off per input
# ----------------------------------------------------------------------------------------------
class Router:
    """Shared by all adapted layers of a model: the route of every input of the current batch."""
    route = None                                             # None: every adapter on


class RoutedLoRA(nn.Module):
    """A linear layer plus one low-rank adapter per task: y = W x + (alpha / r) * sum_k on_k(x) B_k A_k x."""

    def __init__(self, base, router, rank=8, alpha=16):
        super().__init__()
        self.base, self.router, self.rank, self.scale = base, router, rank, alpha / rank
        linear = isinstance(base, nn.Linear)                 # the other kind (GPT-2's Conv1D) stores the weight as in x out
        self.n_in = base.in_features if linear else base.weight.shape[0]
        self.n_out = base.out_features if linear else base.weight.shape[1]
        self.A, self.B = nn.ParameterList(), nn.ParameterList()

    def new_task(self):
        for p in list(self.A) + list(self.B):
            p.requires_grad_(False)                          # older adapters are never changed again
        ref = self.base.weight
        a = torch.empty(self.rank, self.n_in, device=ref.device, dtype=ref.dtype)
        nn.init.kaiming_uniform_(a, a=math.sqrt(5))
        self.A.append(nn.Parameter(a))
        self.B.append(nn.Parameter(torch.zeros(self.n_out, self.rank, device=ref.device, dtype=ref.dtype)))

    def forward(self, x):
        out = self.base(x)
        route = self.router.route
        for k, (a, b) in enumerate(zip(self.A, self.B)):
            delta = (x @ a.T) @ b.T * self.scale
            if route is not None:                            # adapter k+1 is on for inputs whose route is at least k+1
                on = (route >= k + 1).to(delta.dtype).view(-1, *([1] * (delta.dim() - 1)))
                assert on.shape[0] == delta.shape[0], "one route per input of the batch is needed"
                delta = delta * on
            out = out + delta
        return out


ATTENTION_NAMES = ("q", "v", "q_proj", "v_proj", "query", "value", "c_attn")


def add_adapters(model, router, rank=8, alpha=16, names=ATTENTION_NAMES):
    """Freeze the model and wrap its attention projections. Returns the wrapped layers."""
    for p in model.parameters():
        p.requires_grad_(False)
    wrapped = []
    for parent in list(model.modules()):
        for name, child in list(parent.named_children()):
            if name in names and hasattr(child, "weight") and child.weight.dim() == 2 and not isinstance(child, RoutedLoRA):
                layer = RoutedLoRA(child, router, rank, alpha)
                setattr(parent, name, layer)
                wrapped.append(layer)
    assert wrapped, "no attention projection found; pass the layer names of this model in `names`"
    return wrapped


# ----------------------------------------------------------------------------------------------
# 3. A model that learns tasks one after another
# ----------------------------------------------------------------------------------------------
class ContinualModel:
    def __init__(self, model, tokenizer, unrelated_tasks, kappa=0.5, a=1e-3, rank=8, alpha=16, max_len=256, max_new=32, rule="certified"):
        """unrelated_tasks: a few lists of input texts of tasks that will never be learned (they set the scale).
        rule: "certified" or "exact_first" (see the top of this file)."""
        assert rule in ("certified", "exact_first")
        self.model, self.tok, self.max_len, self.max_new, self.rule = model, tokenizer, max_len, max_new, rule
        self.seq2seq = bool(getattr(model.config, "is_encoder_decoder", False))
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token
        self.router = Router()
        self.layers = add_adapters(model, self.router, rank, alpha)
        self.zones = Zones(model.get_input_embeddings().weight, tokenizer.pad_token_id, kappa, a)
        self.zones.fit_background([self.ids(texts) for texts in unrelated_tasks])
        self.device = next(model.parameters()).device

    def ids(self, texts):
        return self.tok(list(texts), max_length=self.max_len, truncation=True)["input_ids"]

    def _pad(self, rows, left=False):
        width = max(len(r) for r in rows)
        ids = torch.full((len(rows), width), self.tok.pad_token_id, dtype=torch.long)
        att = torch.zeros((len(rows), width), dtype=torch.long)
        for i, r in enumerate(rows):
            sl = slice(width - len(r), width) if left else slice(0, len(r))
            ids[i, sl] = torch.tensor(r)
            att[i, sl] = 1
        return ids.to(self.device), att.to(self.device)

    def learn(self, inputs, targets, epochs=3, batch=16, lr=1e-3, seed=0, log=None):
        """Learn one new task. Returns the number of training inputs that an older zone had already claimed."""
        rows = self.ids(inputs)
        blocked = self.zones.add_task(rows)
        for layer in self.layers:
            layer.new_task()
        route = self.zones.route(rows, self.rule)[0]         # inputs that stay with an older task do not train the new adapter
        answers = self.tok(list(targets), max_length=self.max_new, truncation=True, add_special_tokens=self.seq2seq)["input_ids"]
        if not self.seq2seq:
            answers = [a + [self.tok.eos_token_id] for a in answers]
        params = [p for p in self.model.parameters() if p.requires_grad]
        opt = torch.optim.AdamW(params, lr=lr)
        gen = torch.Generator().manual_seed(seed)
        self.model.train()
        step = 0
        for _ in range(epochs):
            order = torch.randperm(len(rows), generator=gen).tolist()
            for s in range(0, len(order), batch):
                pick = order[s:s + batch]
                self.router.route = route[pick].to(self.device)
                if self.seq2seq:
                    ids, att = self._pad([rows[i] for i in pick])
                    lab, lab_att = self._pad([answers[i] for i in pick])
                    loss = self.model(input_ids=ids, attention_mask=att, labels=lab.masked_fill(lab_att == 0, -100)).loss
                else:
                    ids, att = self._pad([rows[i] + answers[i] for i in pick])
                    lab = ids.masked_fill(att == 0, -100)
                    for j, i in enumerate(pick):
                        lab[j, :len(rows[i])] = -100         # the loss is on the answer only
                    loss = self.model(input_ids=ids, attention_mask=att, labels=lab).loss
                loss.backward()
                opt.step()
                opt.zero_grad()
                step += 1
                if log and step % log == 0:
                    print(f"      step {step}: loss {loss.item():.3f}", flush=True)
        self.router.route = None
        return blocked

    @torch.no_grad()
    def answer(self, inputs, rule=None, route=None, batch=16):
        """Returns the answers, the route of every input and whether it is certified. rule: the model's own rule
        if not given; any rule of Zones.route can be asked for, to compare."""
        rows = self.ids(inputs)
        if route is None:
            route, cert = self.zones.route(rows, rule or self.rule)
        else:
            cert = torch.zeros(len(rows), dtype=torch.bool)
        self.model.eval()
        out = []
        for s in range(0, len(rows), batch):
            part = rows[s:s + batch]
            ids, att = self._pad(part, left=not self.seq2seq)
            self.router.route = route[s:s + batch].to(self.device)
            gen = self.model.generate(input_ids=ids, attention_mask=att, max_new_tokens=self.max_new, do_sample=False,
                                      num_beams=1, pad_token_id=self.tok.pad_token_id)
            if not self.seq2seq:
                gen = gen[:, ids.shape[1]:]
            out += [t.strip() for t in self.tok.batch_decode(gen, skip_special_tokens=True)]
        self.router.route = None
        return out, route, cert
'''

In [ ]:
FILES = globals().get("FILES", {})
FILES["demo_no_instruction.py"] = r'''"""Continual learning WITHOUT instructions (Pilot B of silent_adapters/PLAN.md), t5-small, CPU.

Same five tasks and adapters as demo.py, but the definition is removed from every input (empty definition in the template).
Two ways to use the same adapters:
  all    every adapter on
  zones  each task claims a zone in a frozen feature space of the whole input (rule "centre" feature); an input is answered
         with the adapters 1..route, route = oldest task whose zone holds it, else smallest score / threshold.
"""
import argparse, json, math, os, time
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

from certified_switch import ContinualModel

p = argparse.ArgumentParser()
p.add_argument("--model", default="t5-small")
p.add_argument("--data", default="CL_Benchmark")
p.add_argument("--train", type=int, default=400)
p.add_argument("--test", type=int, default=50)
p.add_argument("--epochs", type=int, default=5)
p.add_argument("--lr", type=float, default=5e-3)
p.add_argument("--seed", type=int, default=0)
p.add_argument("--out", default="demo_results")
p.add_argument("--tag", default="")
args = p.parse_args()
torch.manual_seed(args.seed)

TASKS = ["task1510_evalution_relation_extraction", "task363_sst2_polarity_classification",
         "task748_glucose_reverse_cause_event_detection", "task1687_sentiment140_classification",
         "task875_emotion_classification"]
UNRELATED = "mnli,cb,qqp,rte,dbpedia,agnews,yahoo,multirc,boolq,wic".split(",")
TEMPLATE = "Definition: {d}\n\nNow complete the following example -\nInput: {x}\nOutput: "
HERE = os.path.dirname(os.path.abspath(__file__))
os.makedirs(os.path.join(HERE, args.out), exist_ok=True)
stem = os.path.join(HERE, args.out, f"no_instruction_{args.model.replace('/', '_')}_seed{args.seed}{args.tag}")
logf = open(stem + ".log", "w")
ALPHA, M_CAL, KNN = 0.05, 100, 10


def log(*a):
    s = " ".join(str(x) for x in a)
    print(s, flush=True)
    logf.write(s + "\n"); logf.flush()


def read(folder, name, split):
    data = json.load(open(os.path.join(args.data, folder, name, split + ".json")))
    return [(i["input"], i["output"] if isinstance(i["output"], list) else [i["output"]]) for i in data["Instances"]]


def prompt(x):                      # the definition is removed
    return TEMPLATE.replace("{d}", "").replace("{x}", x)


def rouge_l(pred, refs):
    def lcs(a, b):
        row = [0] * (len(b) + 1)
        for x in a:
            prev = 0
            for j, y in enumerate(b):
                prev, row[j + 1] = row[j + 1], prev + 1 if x == y else max(row[j + 1], row[j])
        return row[-1]
    best, a = 0.0, pred.lower().split()
    for ref in refs:
        b = ref.lower().split()
        n = lcs(a, b)
        if n:
            pr, rc = n / len(a), n / len(b)
            best = max(best, 2 * pr * rc / (pr + rc))
    return 100.0 * best


tok = AutoTokenizer.from_pretrained(args.model)
model = AutoModelForSeq2SeqLM.from_pretrained(args.model)
unrelated = [[prompt(x) for x, _ in read("Long_Sequence", n, "train")[:200]] for n in UNRELATED]
cm = ContinualModel(model, tok, unrelated)
Z = cm.zones
log(f"{args.model}: no instruction, settings {vars(args)}")


def feats(texts):
    return (Z._mean(cm.ids(texts)) - Z.mu_all) @ Z.w_all


def knn_score(f, fit):
    return torch.cdist(f, fit).topk(KNN, dim=1, largest=False).values[:, -1]


def train_task(texts, targets):
    """Train the newest adapter on all inputs (route = the new task for every input)."""
    rows = cm.ids(texts)
    for layer in cm.layers:
        layer.new_task()
    k = len(zones)
    ans = tok(list(targets), max_length=cm.max_new, truncation=True)["input_ids"]
    params = [q for q in model.parameters() if q.requires_grad]
    opt = torch.optim.AdamW(params, lr=args.lr)
    gen = torch.Generator().manual_seed(args.seed)
    model.train()
    for _ in range(args.epochs):
        order = torch.randperm(len(rows), generator=gen).tolist()
        for s in range(0, len(order), 16):
            pick = order[s:s + 16]
            cm.router.route = torch.full((len(pick),), k, dtype=torch.long)
            ids, att = cm._pad([rows[i] for i in pick])
            lab, la = cm._pad([ans[i] for i in pick])
            loss = model(input_ids=ids, attention_mask=att, labels=lab.masked_fill(la == 0, -100)).loss
            loss.backward(); opt.step(); opt.zero_grad()
    cm.router.route = None


zones = []          # per task: {"fit": tensor, "q": float}


def scores(f, T):   # n x T
    return torch.stack([knn_score(f, zones[k]["fit"]) for k in range(T)], 1)


def route_of(f, T):
    S = scores(f, T)
    q = torch.tensor([zones[k]["q"] for k in range(T)], dtype=S.dtype)
    inside = S <= q
    first = inside.int().argmax(1)              # oldest zone holding it
    near = (S / q).argmin(1)
    return torch.where(inside.any(1), first, near) + 1, inside


tests, test_f, own_zone, stats = [], [], [], []
first = {"all": [], "zones": []}
per_stage = []
t0 = time.time()
for j, name in enumerate(TASKS):
    train = read("SuperNI", name, "train")[:args.train]
    test = read("SuperNI", name, "test")[:args.test]
    ttexts = [prompt(x) for x, _ in train]
    f_tr = feats(ttexts)
    g = torch.Generator().manual_seed(args.seed)
    perm = torch.randperm(len(train), generator=g)
    m = min(M_CAL, len(train) // 4)
    cal, fit = f_tr[perm[:m]], f_tr[perm[m:]]
    srt = knn_score(cal, fit).sort().values
    idx = min(math.ceil((1 - ALPHA) * (m + 1)), m) - 1
    zones.append({"fit": fit, "q": float(srt[idx])})
    f_te = feats([prompt(x) for x, _ in test])
    blocked_tr = blocked_te = 0
    if j:
        blocked_tr = int(torch.stack([knn_score(f_tr, zones[k]["fit"]) <= zones[k]["q"] for k in range(j)], 1).any(1).sum())
        blocked_te = int(torch.stack([knn_score(f_te, zones[k]["fit"]) <= zones[k]["q"] for k in range(j)], 1).any(1).sum())
    cover = knn_score(f_te, zones[j]["fit"]) <= zones[j]["q"]
    stats.append({"task": name.split("_")[0], "q": zones[j]["q"], "blocked_train": blocked_tr, "blocked_test": blocked_te,
                  "coverage_test": int(cover.sum()), "n_train": len(train), "n_test": len(test)})
    train_task(ttexts, [y[0] for _, y in train])
    tests.append(([prompt(x) for x, _ in test], [y for _, y in test]))
    test_f.append(f_te)
    own_zone.append(cover)
    T = j + 1
    share = [float((route_of(test_f[jj], T)[0] == jj + 1).float().mean()) for jj in range(T)]
    per_stage.append(share)
    first["all"].append(cm.answer(tests[j][0], route=torch.full((len(test),), T, dtype=torch.long))[0])
    first["zones"].append(cm.answer(tests[j][0], route=route_of(test_f[j], T)[0])[0])
    log(f"task {T} learned ({name.split('_')[0]}): q={zones[j]['q']:.3f}, train inside older zone {blocked_tr}/{len(train)}, "
        f"test inside older zone {blocked_te}/{len(test)}, coverage {int(cover.sum())}/{len(test)}, "
        f"routed to own task {[round(s, 2) for s in share]}; {(time.time() - t0) / 60:.1f} min")

T = len(TASKS)
res, final_ans = {}, {}
for rule in ("all", "zones"):
    learned = final = changed = total = 0
    forget, per_task_first, per_task_final = [], [], []
    for j in range(T):
        texts, refs = tests[j]
        r = torch.full((len(texts),), T, dtype=torch.long) if rule == "all" else route_of(test_f[j], T)[0]
        ans = cm.answer(texts, route=r)[0]
        final_ans[rule, j] = (ans, r)
        s0 = sum(rouge_l(a, y) for a, y in zip(first[rule][j], refs)) / len(refs)
        s1 = sum(rouge_l(a, y) for a, y in zip(ans, refs)) / len(refs)
        per_task_first.append(s0); per_task_final.append(s1)
        learned += s0 / T; final += s1 / T
        if j < T - 1:
            forget.append(s0 - s1)
            changed += sum(a != b for a, b in zip(first[rule][j], ans)); total += len(ans)
    res[rule] = {"learned": learned, "AP": final, "FT": sum(forget) / len(forget), "changed": changed, "compared": total,
                 "score_after_learning_per_task": per_task_first, "score_final_per_task": per_task_final}
    log(f"{rule:<6} learned {learned:6.2f}  at the end {final:6.2f}  forgetting {res[rule]['FT']:6.2f}  answers changed {changed} of {total}")
    log(f"       per task after learning {[round(s, 1) for s in per_task_first]}  at the end {[round(s, 1) for s in per_task_final]}")

in_own = chg_in_own = 0
for j in range(T - 1):
    ans, _ = final_ans["zones", j]
    for i in range(len(ans)):
        if bool(own_zone[j][i]):
            in_own += 1
            chg_in_own += int(ans[i] != first["zones"][j][i])
assert chg_in_own == 0, f"{chg_in_own} answers inside their own zone changed"
res["zones"].update({"inside_own_zone_first4": in_own, "changed_inside_own_zone": chg_in_own,
                     "routed_to_own_task_after_each_task": per_stage,
                     "routed_to_own_task_final_per_task": per_stage[-1],
                     "routed_to_own_task_final_mean": sum(per_stage[-1]) / T})
log(f"zones: inside own zone (tasks 1-4) {in_own} of {sum(len(tests[j][0]) for j in range(T - 1))}, changed among them {chg_in_own}; "
    f"routed to own task at end {[round(s, 2) for s in per_stage[-1]]}")
log("per task stats:", json.dumps(stats))

z, a = res["zones"], res["all"]
if z["changed"] > 40:
    verdict = "does not"
elif z["changed"] <= 20 and z["FT"] <= 3 and a["changed"] > 100:
    verdict = "works"
else:
    verdict = "partial"
log("verdict:", verdict)
json.dump({"model": args.model, "args": vars(args), "instruction": "removed from every input", "alpha": ALPHA, "m_cal": M_CAL,
           "knn": KNN, "rules": res, "per_task": stats, "verdict": verdict, "minutes": (time.time() - t0) / 60},
          open(stem + ".json", "w"), indent=1)
log(f"done in {(time.time() - t0) / 60:.1f} min")
'''

In [ ]:
for name, text in FILES.items():
    with open(os.path.join(CODE, name), "w") as f:
        f.write(text)
print("written to", CODE, ":", ", ".join(f"{n} ({len(t.splitlines())} lines)" for n, t in FILES.items()))
if "cuda" not in FILES["demo_no_instruction.py"] and "device" not in FILES["demo_no_instruction.py"]:
    print("NOTE: demo_no_instruction.py never mentions cuda/device, so each job runs on the CPU of the machine unless the script is changed.")

## Machine, data, model

In [ ]:
NO_INTERNET = ("NO INTERNET: the notebook cannot download what it needs ({what}). On Kaggle the internet is off until you switch it on: panel on "
               "the right -> Session options (or Notebook options) -> Internet -> on. The switch appears only after your phone number is "
               "verified (kaggle.com -> Settings -> Phone verification). Then run all again.")

gpu_lines = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip().splitlines()
if not NB_TEST:
    assert gpu_lines, "No GPU. On Kaggle: panel on the right -> Session options -> Accelerator -> GPU T4 x2. Then run all again."
N_GPUS = len(gpu_lines)
print(f"GPUs seen: {N_GPUS}" + ("" if not gpu_lines else " -> " + " | ".join(gpu_lines)))
if not NB_TEST and N_GPUS < len(CFG["seeds"]):
    print(f"only {N_GPUS} GPU(s): the {len(CFG['seeds'])} jobs will share them (the second one starts when the first has finished if there is 1 GPU)")
WORKERS = max(1, min(N_GPUS, len(CFG["seeds"]))) if not NB_TEST else 1

DATA = os.environ.get("NI_NB_DATA")
if not DATA:
    repo = os.path.join(WORK, "gainlora")
    if not os.path.isdir(os.path.join(repo, "CL_Benchmark", "SuperNI")):
        shutil.rmtree(repo, ignore_errors=True)
        r = subprocess.run(["git", "clone", CFG["repo_url"], repo], capture_output=True, text=True)
        if r.returncode != 0:
            import socket
            try:
                socket.create_connection(("github.com", 443), timeout=10).close()
            except OSError:
                raise RuntimeError(NO_INTERNET.format(what="git clone " + CFG["repo_url"])) from None
            raise RuntimeError("git clone failed although github.com can be reached:\n" + (r.stdout + r.stderr)[-2000:])
        subprocess.run(["git", "-C", repo, "checkout", CFG["repo_commit"]], check=True, capture_output=True, text=True)
    DATA = os.path.join(repo, "CL_Benchmark")
TASKS = ["task1510_evalution_relation_extraction", "task363_sst2_polarity_classification", "task748_glucose_reverse_cause_event_detection",
         "task1687_sentiment140_classification", "task875_emotion_classification"]
for t in TASKS:
    for split in ("train", "test"):
        assert os.path.exists(os.path.join(DATA, "SuperNI", t, split + ".json")), f"missing data file {t}/{split}.json in {DATA}"
for t in "mnli,cb,qqp,rte,dbpedia,agnews,yahoo,multirc,boolq,wic".split(","):
    assert os.path.exists(os.path.join(DATA, "Long_Sequence", t, "train.json")), f"missing Long_Sequence/{t}/train.json in {DATA}"
print("data:", DATA)

# the model and tokenizer, downloaded once here (so that the parallel jobs do not download at the same time)
check = ("import torch, transformers; from transformers import AutoModelForSeq2SeqLM, AutoTokenizer; "
         f"AutoTokenizer.from_pretrained('{CFG['model']}'); AutoModelForSeq2SeqLM.from_pretrained('{CFG['model']}'); "
         "print('torch', torch.__version__, '| transformers', transformers.__version__, '| cuda', torch.cuda.is_available())")
r = subprocess.run([sys.executable, "-c", check], capture_output=True, text=True)
if r.returncode != 0 and "sentencepiece" in (r.stdout + r.stderr).lower():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentencepiece"], capture_output=True, text=True)
    r = subprocess.run([sys.executable, "-c", check], capture_output=True, text=True)
if r.returncode != 0:
    text = r.stdout + r.stderr
    if any(w in text for w in ("ConnectionError", "Max retries", "Temporary failure", "Name or service", "offline", "LocalEntryNotFound", "couldn't connect", "Can't load")):
        raise RuntimeError(NO_INTERNET.format(what=f"the {CFG['model']} weights from huggingface.co") + "\n\nDetails:\n" + text[-1500:])
    raise RuntimeError("loading the model failed:\n" + text[-3000:])
print(r.stdout.strip())

## The runs
One process per seed, one GPU each (`CUDA_VISIBLE_DEVICES`), one log file each in `logs/`. The last line of each log is printed every minute.

In [ ]:
FAILED, RUNNING = {}, {}
lock = threading.Lock()
queue_ = list(CFG["seeds"])


def tail(path, n=1):
    try:
        with open(path, "rb") as f:
            f.seek(0, 2); f.seek(max(0, f.tell() - 6000))
            lines = [l for l in f.read().decode(errors="replace").splitlines() if l.strip()]
        return lines[-n:]
    except OSError:
        return []


def run_job(seed, gpu):
    log_path = os.path.join(LOGS, f"seed{seed}.log")
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), TOKENIZERS_PARALLELISM="false", PYTHONUNBUFFERED="1", HF_HUB_DISABLE_PROGRESS_BARS="1")
    cmd = [sys.executable, os.path.join(CODE, "demo_no_instruction.py"), "--model", CFG["model"], "--data", DATA, "--train", str(CFG["train"]),
           "--test", str(CFG["test"]), "--epochs", str(CFG["epochs"]), "--lr", CFG["lr"], "--seed", str(seed), "--out", RESULTS, "--tag", "gpu"]
    t0 = time.time()
    with open(log_path, "a") as log:
        proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env, cwd=CODE)
        with lock:
            RUNNING[seed] = (proc, log_path, t0, gpu)
        code = proc.wait()
    with lock:
        RUNNING.pop(seed, None)
        if code != 0:
            FAILED[seed] = code
    print(f"[GPU {gpu}] seed {seed}: {'finished' if code == 0 else 'FAILED (exit code %d)' % code} after {(time.time() - t0) / 60:.1f} min", flush=True)
    if code != 0:
        print("   the end of its log:\n   " + "\n   ".join(tail(log_path, 40)), flush=True)


def worker(gpu):
    while True:
        with lock:
            if not queue_:
                return
            seed = queue_.pop(0)
        run_job(seed, gpu)


def reporter(stop):
    while not stop.wait(CFG["progress_every_s"]):
        with lock:
            now = list(RUNNING.items())
        for seed, (proc, path, t0, gpu) in now:
            last = tail(path, 2)
            print(f"  ... GPU {gpu}, seed {seed}, {(time.time() - t0) / 60:.0f} min: " + (" | ".join(l[:160] for l in last) if last else "(no output yet)"), flush=True)


print(f"starting {len(CFG['seeds'])} job(s) on {WORKERS} worker(s)", flush=True)
stop = threading.Event()
threading.Thread(target=reporter, args=(stop,), daemon=True).start()
threads = [threading.Thread(target=worker, args=(g,)) for g in range(WORKERS)]
for t in threads:
    t.start()
for t in threads:
    t.join()
stop.set()

## Results
Per rule: **learned** = mean score of each task right after it was learned; **at the end** = mean score after the last task; **forgetting** = mean drop
of tasks 1 to 4; **answers changed** = test inputs of tasks 1 to 4 whose answer at the end differs from the one right after learning. Scores are Rouge-L (0 to 100).

In [ ]:
rows = []
for seed in CFG["seeds"]:
    p = os.path.join(RESULTS, f"no_instruction_{CFG['model'].replace('/', '_')}_seed{seed}gpu.json")
    if not os.path.exists(p):
        print(f"seed {seed}: no result file ({p})")
        continue
    d = json.load(open(p))
    for rule in ("all", "zones"):
        x = d["rules"][rule]
        rows.append((seed, rule, x["learned"], x["AP"], x["FT"], f"{x['changed']} of {x['compared']}", d["verdict"] if rule == "zones" else ""))
print(f"{'seed':<6}{'rule':<8}{'learned':>9}{'at the end':>12}{'forgetting':>12}{'answers changed':>18}  verdict")
for s, rule, a, b, c, ch, v in rows:
    print(f"{s:<6}{rule:<8}{a:9.2f}{b:12.2f}{c:12.2f}{ch:>18}  {v}")
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:       # small result files only; no model weights anywhere
    for root, _, files in os.walk(OUT):
        for f in files:
            if f.endswith((".json", ".log")):
                z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), os.path.dirname(OUT.rstrip("/"))))
print("zip with all .json and .log files:", ZIP, f"({os.path.getsize(ZIP) / 1e6:.2f} MB)")
if FAILED:
    print("\nJOBS THAT FAILED:", {s: c for s, c in FAILED.items()}, "- logs in", LOGS)
else:
    print("\nall jobs finished")